# Test notebook for DPD funcionality

In [1]:
import warnings

warnings.filterwarnings("ignore")

In [2]:
from flowermd.library import PPS

molecules = PPS(num_mols=30, lengths=30)
molecules.coarse_grain(beads={"A": "c1cc(S)ccc1"})
molecules.bond_length

Warning on use of the timeseries module: If the inherent timescales of the system are long compared to those being analyzed, this statistical inefficiency may be an underestimate.  The estimate presumes the use of many statistically independent samples.  Tests should be performed to assess whether this condition is satisfied.   Be cautious in the interpretation of the data.

****** PyMBAR will use 64-bit JAX! *******
* JAX is currently set to 32-bit bitsize *
* which is its default.                  *
*                                        *
* PyMBAR requires 64-bit mode and WILL   *
* enable JAX's 64-bit mode when called.  *
*                                        *
* This MAY cause problems with other     *
* Uses of JAX in the same code.          *
******************************************



0.176

## New Random Walk System Class

In [3]:
from flowermd.library import RandomWalk
import unyt as u

system = RandomWalk(
    molecules=molecules,
    density=1.32 * u.Unit("g/cm**3"),
    bond_length=1.4226,
    buffer=0.58,
)

(900, 3)


In [4]:
system.box.Lx

np.float64(4.966861)

## FF from GMSO XML file

In [5]:
from flowermd.base.forcefield import BaseXMLForcefield
from flowermd.assets import FF_DIR
class Bead_Spring_DPD(BaseXMLForcefield):
    """Forcefield class for loading a forcefield from an XML file."""

    def __init__(self, forcefield_files=f"{FF_DIR}/hoomd-dpd-hhp.xml"):
        self.gmso_xml = True
        super(Bead_Spring_DPD, self).__init__(forcefield_files=forcefield_files)
        self.description = "DPD forcefield loaded from an XML file."

In [6]:
system.apply_forcefield(force_field=Bead_Spring_DPD(),r_cut=1.15)

DocumentInvalid: Element 'FFMetaData': This element is not expected., line 2

## Forcefield class

In [ ]:
from flowermd.library import DPD

A = 2000
gamma = 1500
kT = 1.0
r_cut = 1.4226
bond_k = 50000
bond_r0 = 1.4226
dpd_ff = DPD(
    A=A, gamma=gamma, kT=kT, r_cut=r_cut, bond_k=bond_k, bond_r0=bond_r0
)

In [ ]:
from flowermd.library.simulations.dpd_init import DPDInit

sim = DPDInit(
    initial_state=system.hoomd_snapshot,
    forcefield=dpd_ff.hoomd_forces,
    gsd_write_freq=100,
    log_write_freq=100,
    A=A,
    r_cut=r_cut,
    r=1.2,
    N=molecules.n_particles,
    box=system.box.Lx,
    sim_steps_incr=100,
)

In [ ]:
import hoomd

for writer in sim.operations.writers:
    if isinstance(writer, hoomd.write.GSD):
        writer.flush()

In [ ]:
#system.to_gsd("random_walk_test.gsd")